# DCA Buy-the-Bear — ETHUSDT Monthly (Progressive Reinvestment)

Same optimum strategy on ETHUSDT: $10/red month + progressive reinvestment (1% start, +2%/buy, cap 50%), sell 70% at new ATH.

In [ ]:
import pandas as pd
import requests
import time
import json
import numpy as np
from datetime import datetime, timezone

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: '%.6f' % x)

In [ ]:
# Fetch ETHUSDT monthly klines

def fetch_monthly_klines(symbol='ETHUSDT', limit=500):
    url = 'https://api.binance.com/api/v3/klines'
    all_klines = []
    start_time = 0
    while True:
        params = {'symbol': symbol, 'interval': '1M', 'startTime': start_time, 'limit': limit}
        resp = requests.get(url, params=params)
        data = resp.json()
        if not data or len(data) == 0:
            break
        all_klines.extend(data)
        print(f'Fetched {len(data)} months, up to {datetime.fromtimestamp(data[-1][0]/1000, tz=timezone.utc).strftime("%Y-%m")}')
        if len(data) < limit:
            break
        start_time = data[-1][0] + 1
        time.sleep(0.1)
    return all_klines

klines = fetch_monthly_klines()

In [ ]:
columns = ['timestamp', 'open', 'high', 'low', 'close', 'volume',
           'close_time', 'quote_vol', 'trades', 'taker_buy_base', 'taker_buy_quote', 'ignore']
df = pd.DataFrame(klines, columns=columns)
df['date'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
for c in ['open', 'high', 'low', 'close', 'volume']:
    df[c] = df[c].astype(float)
df = df[['date', 'open', 'high', 'low', 'close', 'volume']].copy()
df = df.sort_values('date').reset_index(drop=True)
print(f'{df["date"].min().strftime("%Y-%m")} → {df["date"].max().strftime("%Y-%m")} ({len(df)} months)')

In [ ]:
# Progressive reinvestment strategy (optimum parameters)

START_PCT = 1.0; INCREMENT = 2.0; CAP = 50.0; BASE = 10.0

btc_held = 0.0; usdt_reserve = 0.0; total_invested = 0.0; highest_close = 0.0
reinvest_pct = START_PCT

records = []

for i, row in df.iterrows():
    close = row['close']
    month = row['date']
    action = 'nothing'

    if close < row['open']:
        extra = usdt_reserve * (reinvest_pct / 100.0) if usdt_reserve > 0 else 0.0
        total_usdt = BASE + extra
        btc_bought = total_usdt / close
        btc_held += btc_bought
        total_invested += BASE
        usdt_reserve -= extra
        action = f'buy ${total_usdt:.1f} @ {close:.1f} (${BASE:.0f}+{reinvest_pct:.0f}%={extra:.1f})'

    prev_highest = highest_close
    if close > highest_close:
        highest_close = close

    if btc_held > 0.000001 and close > prev_highest:
        sell_coin = btc_held * 0.7
        sell_usdt = sell_coin * close
        btc_held -= sell_coin
        usdt_reserve += sell_usdt
        reinvest_pct = START_PCT
        act = f'sell 70% @ {close:.1f} (-{sell_coin:.6f} ETH, +{sell_usdt:.2f} USDT)'
        action = f'{action} | {act}' if 'buy' in action else act
    else:
        if reinvest_pct < CAP:
            reinvest_pct = min(CAP, reinvest_pct + INCREMENT)

    portfolio_value = btc_held * close + usdt_reserve
    coin_value = btc_held * close

    records.append({
        'date': month, 'close': close, 'action': action,
        'reinvest_pct': reinvest_pct,
        'coin_held': btc_held, 'coin_value': coin_value,
        'usdt_reserve': usdt_reserve, 'total_invested': total_invested,
        'portfolio_value': portfolio_value,
    })

results = pd.DataFrame(records)
print(f'Total months: {len(results)}')
print(f'Trades with action: {len(results[results["action"] != "nothing"])}')

In [ ]:
# Metrics

final = results.iloc[-1]
last_close = final['close']
total_pnl = final['portfolio_value'] - final['total_invested']
ret_pct = (final['portfolio_value'] / final['total_invested'] - 1) * 100

eq = results['portfolio_value']
running_max = eq.cummax()
dd_raw = (running_max - eq) / running_max
max_dd = dd_raw.max() * 100

monthly_returns = eq.pct_change().dropna()
monthly_returns = monthly_returns[monthly_returns.index >= 12]
sharpe = (monthly_returns.mean() / monthly_returns.std()) * np.sqrt(12) if monthly_returns.std() > 0 else 0.0

pos_sum = monthly_returns[monthly_returns > 0].sum()
neg_sum = monthly_returns[monthly_returns < 0].abs().sum()
pf = pos_sum / neg_sum if neg_sum > 0 else float('inf')

annualized_ret = (1 + ret_pct / 100) ** (12 / len(df)) - 1
calmar = annualized_ret / (max_dd / 100) if max_dd > 0 else 0

buys = [r for r in records if 'buy' in r['action']]
sells = [r for r in records if 'sell' in r['action']]

print('='*65)
print('PORTFOLIO SUMMARY — ETHUSDT Progressive')
print('='*65)
print(f'Parameters:            start={START_PCT}%, +{INCREMENT}%/buy, cap={CAP}%')
print(f'Total invested:        {final["total_invested"]:.2f} USDT')
print(f'ETH held:              {final["coin_held"]:.8f} ETH')
print(f'ETH value:             {final["coin_value"]:.2f} USDT (at {last_close:.2f})')
print(f'USDT reserve:          {final["usdt_reserve"]:.2f} USDT')
print(f'Portfolio value:       {final["portfolio_value"]:.2f} USDT')
print(f'Total P&L:             {total_pnl:.2f} USDT')
print(f'Return:                {ret_pct:.2f}%')
print(f'Max drawdown:          {max_dd:.2f}%')
print(f'Sharpe ratio:          {sharpe:.2f}')
print(f'Profit factor:         {pf:.2f}')
print(f'Calmar ratio:          {calmar:.2f}')
print(f'Months:                {len(results)}')
print(f'Buy months:            {len(buys)}')
print(f'Sell months:           {len(sells)}')
print()

# Compare with ETH buy & hold on red candles only
eth_inv = 0; eth_held = 0
for i, row in df.iterrows():
    if row['close'] < row['open']:
        eth_held += BASE / row['close']
        eth_inv += BASE
eth_hodl = eth_held * last_close
eth_ret = (eth_hodl / eth_inv - 1) * 100

print(f'ETH Red-only HODL:     ${eth_hodl:.2f} ({eth_ret:.1f}% return)')

# Compare with classic monthly DCA
dca_held = sum(BASE / row['close'] for _, row in df.iterrows())
dca_inv = BASE * len(df)
dca_val = dca_held * last_close
dca_ret = (dca_val / dca_inv - 1) * 100

print(f'ETH Monthly DCA:       ${dca_val:.2f} ({dca_ret:.1f}% return)')

In [ ]:
# Chart

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(3, 1, figsize=(14, 9), gridspec_kw={'height_ratios': [3, 1, 1]})

ax = axes[0]
ax.fill_between(results['date'], results['total_invested'], results['portfolio_value'],
                where=results['portfolio_value'] >= results['total_invested'],
                color='green', alpha=0.12, label='Profit')
ax.fill_between(results['date'], results['portfolio_value'], results['total_invested'],
                where=results['portfolio_value'] < results['total_invested'],
                color='red', alpha=0.12, label='Loss')
ax.plot(results['date'], results['total_invested'], color='gray', linewidth=1.5, linestyle='--', label='Total Invested')
ax.plot(results['date'], results['portfolio_value'], color='#2563eb', linewidth=2, label='Portfolio Value')

buys_df = results[results['action'].str.contains('buy', na=False)]
sells_df = results[results['action'].str.contains('sell', na=False)]
ax.scatter(buys_df['date'], buys_df['portfolio_value'], color='#16a34a', s=25, marker='^', zorder=5, alpha=0.7)
ax.scatter(sells_df['date'], sells_df['portfolio_value'], color='#dc2626', s=50, marker='v', zorder=5, label='Sell 70%')

ax.set_title('DCA Buy-the-Bear — ETHUSDT Monthly (Progressive Reinvestment)', fontsize=13, fontweight='bold')
ax.set_ylabel('USDT')
ax.legend(loc='upper left', fontsize=8, ncol=3)
ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax = axes[1]
ax.fill_between(results['date'], 0, dd_raw * 100, color='#dc2626', alpha=0.35)
ax.plot(results['date'], dd_raw * 100, color='#dc2626', linewidth=0.8)
ax.axhline(y=-max_dd, color='#991b1b', linestyle=':', linewidth=1.2, alpha=0.8, label=f'Max DD: {max_dd:.1f}%')
ax.set_ylabel('Drawdown (%)')
ax.legend(loc='lower left', fontsize=8)
ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax = axes[2]
ax.fill_between(results['date'], 0, results['coin_value'], color='#8b5cf6', alpha=0.5, label='ETH Value')
ax.fill_between(results['date'], results['coin_value'], results['coin_value'] + results['usdt_reserve'],
                color='#3b82f6', alpha=0.4, label='USDT Reserve')
ax.set_ylabel('USDT')
ax.set_xlabel('Date')
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('dca-bt-buy-bot-eth-progressive.png', dpi=150, bbox_inches='tight')
print('Chart saved as dca-bt-buy-bot-eth-progressive.png')
plt.show()

In [ ]:
# All trades

pd.set_option('display.max_rows', None)
with_actions = results[results['action'] != 'nothing'].copy()
with_actions[['date', 'close', 'reinvest_pct', 'action', 'coin_held', 'usdt_reserve', 'total_invested', 'portfolio_value']]